In [1]:
import re

import pandas as pd
from pandas.api.typing import NAType

In [2]:
# Load dataset
file_name = '../../data/2025-2026/ncsl_ai_db_2025_2026.csv'
df = pd.read_csv(file_name)

In [3]:
# Load 2nd dataset
df = pd.read_csv(file_name)

# Rename column names for easier access
df = df.rename(columns={
    "Bill": "bill_number",
    "Source": "source",
    "Status": "status",
    "Date of Last Action": "date_last_action",
    "Author": "author",
    "Topics": "category",
    "Summary": "summary",
    "History": "history",
})

In [4]:
def extract_year(row) -> str | NAType:
    """Extract year from 'Date of Last Action' or 'History' (backup)"""
    match = re.search(r'(20\d{2})', str(row['date_last_action']))
    if match:
        return match.group(1)

    match_hist = re.search(r'(20\d{2})', str(row['History']))
    if match_hist:
        return match_hist.group(1)

    return pd.NA

In [5]:
# Clean each string column
for col in df.select_dtypes(include=['str']).columns:
    df[col] = df[col].astype(str).str.replace(
        r'\s+', ' ', regex=True).str.strip()
    # Replace string 'nan' back to real NaNs
    df[col] = df[col].replace('nan', pd.NA)

    # Retrieve bill session year
    df['session_year'] = df.apply(extract_year, axis=1)

    # Handle missing/empty bill topics
    df['Topics'] = df['category'].fillna('None')

In [6]:
# Export to csv
file_name = '../../data/2025-2026/ncsl_ai_db_2025_2026_clean.csv'
df.to_csv(file_name, index=False)

df.columns

Index(['bill_number', 'Year', 'Stage', 'status', 'date_last_action', 'author',
       'category', 'summary', 'history', 'session_year', 'Topics'],
      dtype='str')